# 02 – Vet clinics cleaning and normalization (v1)

This notebook takes the v0 dataset (OSM + LOR context) and produces a cleaned
and normalized vet clinics table (v1) ready to be loaded into the unified POI
schema.

Key goals:

- Preserve OSM numeric `id` as the primary identifier.
- Keep `geometry` for the unified POI table.
- Normalise address, contact, and operational fields.
- Use NULLs (not empty strings) for missing data.
- Optionally use Nominatim to backfill missing addresses.
- Flag records with almost no information via `has_minimum_info`.

In [1]:
from pathlib import Path
import os

ROOT = Path.cwd()
if (ROOT / "veterinary_clinics" / "sources").exists():
    os.chdir(ROOT / "veterinary_clinics")
    print("Changed CWD to:", Path.cwd())
else:
    print("Current CWD:", ROOT)
    print("Assuming this notebook already runs inside `veterinary_clinics`.")

import pandas as pd
import numpy as np

V0_PATH = Path("cache/vets_osm_berlin_with_lor_latest_v0.csv")
df = pd.read_csv(V0_PATH)

print("v0 shape:", df.shape)
df.head()

Current CWD: /Users/jorge/Projects/layered-populate-data-pool-da/veterinary_clinics
Assuming this notebook already runs inside `veterinary_clinics`.
v0 shape: (175, 27)


,id,element,source_osm_id,name,addr:street,addr:housenumber,addr:postcode,addr:city,phone,contact:phone,...,wheelchair:description,emergency,lat,lon,geometry,lor_id,district,district_id,neighborhood,neighborhood_id
0,268917040,node,node/268917040,Tierarztpraxis am Urban,Baerwaldstraße,69,10961.0,Berlin,NaN,NaN,...,NaN,NaN,52.495684,13.405233,POINT (13.4052329 52.4956842),re_ortsteil.0202,Friedrichshain-Kreuzberg,11002002,Kreuzberg,202
1,299795048,node,node/299795048,Dr. med. vet. Elke Hartwig,Straße 48,67,13125.0,Berlin,+49 30 9437820,NaN,...,NaN,NaN,52.606286,13.479555,POINT (13.4795548 52.60628629999999),re_ortsteil.0305,Pankow,11003003,Karow,305
2,347294456,node,node/347294456,Tierarztpraxis Dr. Bernhard Sörensen,Königsberger Straße,36,12207.0,Berlin,+49 30 7738321,NaN,...,NaN,NaN,52.429722,13.320133,POINT (13.3201326 52.4297216),re_ortsteil.0602,Steglitz-Zehlendorf,11006006,Lichterfelde,602
3,394867279,node,node/394867279,Tierarztpraxis Jeanette Koepsel,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,52.535199,13.270573,POINT (13.2705734 52.5351995),re_ortsteil.0503,Spandau,11005005,Siemensstadt,503
4,411550894,node,node/411550894,Kleintierarztpraxis Berlin Kaulsdorf,Planitzstraße,19,12621.0,Berlin,+49 30 53018585,NaN,...,NaN,NaN,52.509511,13.589635,POINT (13.5896353 52.50951139999999),re_ortsteil.1003,Marzahn-Hellersdorf,11010010,Kaulsdorf,1003


## 1. Helper functions

We define small helpers for:

- safe string coercion and trimming,
- building a full address from components,
- deriving a simple `operating_days` label from `opening_hours`.

In [2]:
# Cell 1 – Helper functions

def to_str(series: pd.Series) -> pd.Series:
    """Convert to pandas string dtype and strip whitespace, keeping NULLs as <NA>."""
    s = series.astype("string")
    return s.str.strip()

def build_full_address(row) -> str | None:
    parts = []
    if pd.notna(row.get("addr_street")):
        street = row["addr_street"]
        if pd.notna(row.get("addr_housenumber")):
            street = f"{street} {row['addr_housenumber']}"
        parts.append(street)
    if pd.notna(row.get("addr_postcode")) or pd.notna(row.get("addr_city")):
        city_part = " ".join(
            str(x)
            for x in [row.get("addr_postcode"), row.get("addr_city")]
            if pd.notna(x)
        ).strip()
        if city_part:
            parts.append(city_part)
    if not parts:
        return None
    return ", ".join(parts)

def infer_operating_days(opening_hours: str | None) -> str | None:
    """Very simple heuristic based on opening_hours string."""
    if pd.isna(opening_hours):
        return None

    s = opening_hours.lower()
    if "su" in s or "sun" in s:
        return "Mon–Sun"
    if "sa" in s or "sat" in s:
        return "Mon–Sat"
    if any(d in s for d in ["mo", "tu", "we", "th", "fr"]):
        return "Mon–Fri"
    return None

## 2. Build base v1 DataFrame (before reverse geocoding)

We construct the v1 schema without `full_address` yet.  
Address components may be improved later via Nominatim.

In [3]:
df_clean = pd.DataFrame()

# 2.1 Identifier (numeric OSM id as string)
if "id" in df.columns:
    df_clean["id"] = df["id"].astype("Int64").astype("string")
elif "source_osm_id" in df.columns:
    df_clean["id"] = (
        df["source_osm_id"]
        .astype("string")
        .str.extract(r"(\d+)$", expand=False)
    )
else:
    df_clean["id"] = df.index.astype(str)

# 2.2 Clinic name (fallback: name -> operator -> address)
name_raw = to_str(df["name"])
operator_raw = to_str(df["operator"])

addr_street_raw = to_str(df["addr:street"])
addr_housenumber_raw = to_str(df["addr:housenumber"])

fallback_name = addr_street_raw.copy()
has_hn = addr_housenumber_raw.notna()
fallback_name = fallback_name.where(~has_hn, fallback_name + " " + addr_housenumber_raw)

clinic_name = name_raw.copy()
clinic_name = clinic_name.fillna(operator_raw)
clinic_name = clinic_name.fillna(fallback_name)

df_clean["clinic_name"] = clinic_name

# 2.3 Address components
df_clean["addr_street"] = addr_street_raw
df_clean["addr_housenumber"] = addr_housenumber_raw
df_clean["addr_postcode"] = to_str(df["addr:postcode"])
df_clean["addr_city"] = to_str(df["addr:city"])

# 2.4 District / neighborhood / IDs
df_clean["district"] = to_str(df["district"])
df_clean["district_id"] = to_str(df["district_id"])
df_clean["neighborhood"] = to_str(df["neighborhood"])
df_clean["neighborhood_id"] = to_str(df["neighborhood_id"])
df_clean["lor_id"] = to_str(df["lor_id"])

# 2.5 Coordinates and geometry
df_clean["latitude"] = df["lat"]
df_clean["longitude"] = df["lon"]
df_clean["geometry"] = df["geometry"]

# 2.6 Services and operating hours
if "emergency" in df.columns:
    emergency_raw = to_str(df["emergency"])
else:
    # columna no existe: todo NA
    emergency_raw = pd.Series(pd.NA, index=df.index, dtype="string")

# Inicializamos services_offered como todo NULL
services = pd.Series(pd.NA, index=df.index, dtype="string")

# True donde emergency == "yes" (ignorando mayúsculas/minúsculas)
mask_emergency = emergency_raw.str.lower().eq("yes")

# Reemplazamos solo donde la máscara es True; los <NA> se tratan como False
services.loc[mask_emergency.fillna(False)] = "emergency"

df_clean["services_offered"] = services

opening_hours_raw = to_str(df["opening_hours"])
df_clean["operating_hours"] = opening_hours_raw
df_clean["operating_days"] = opening_hours_raw.map(infer_operating_days)

# 2.7 Contact details
phone_raw = to_str(df.get("phone"))
contact_phone_raw = to_str(df.get("contact:phone"))
email_raw = to_str(df.get("email"))
contact_email_raw = to_str(df.get("contact:email"))
website_raw = to_str(df.get("website"))
contact_website_raw = to_str(df.get("contact:website"))

df_clean["phone_main"] = phone_raw.fillna(contact_phone_raw)
df_clean["email_main"] = email_raw.fillna(contact_email_raw)
df_clean["website_main"] = website_raw.fillna(contact_website_raw)

# Combined contact_info (for display)
contact_parts = []

contact_parts.append(
    df_clean["phone_main"].map(lambda x: f"phone: {x}" if pd.notna(x) else None)
)
contact_parts.append(
    df_clean["email_main"].map(lambda x: f"email: {x}" if pd.notna(x) else None)
)
contact_parts.append(
    df_clean["website_main"].map(lambda x: f"web: {x}" if pd.notna(x) else None)
)

contact_info = pd.Series(index=df.index, dtype="string")
for s in contact_parts:
    if contact_info.isna().all():
        contact_info = s
    else:
        contact_info = contact_info.combine(
            s,
            lambda a, b: (
                (a if pd.notna(a) else "") +
                ("; " if pd.notna(a) and pd.notna(b) else "") +
                (b if pd.notna(b) else "")
            ).strip() or None,
        )

df_clean["contact_info"] = contact_info.astype("string")

# Accessibility
wheelchair_raw = to_str(df.get("wheelchair"))
wheelchair_desc = to_str(df.get("wheelchair:description"))

accessibility = wheelchair_raw.copy()
has_desc = wheelchair_desc.notna()
accessibility = accessibility.where(~has_desc, accessibility + " – " + wheelchair_desc)

df_clean["accessibility_features"] = accessibility

df_clean.head()

,id,clinic_name,addr_street,addr_housenumber,addr_postcode,addr_city,district,district_id,neighborhood,neighborhood_id,...,longitude,geometry,services_offered,operating_hours,operating_days,phone_main,email_main,website_main,contact_info,accessibility_features
0,268917040,Tierarztpraxis am Urban,Baerwaldstraße,69,10961.0,Berlin,Friedrichshain-Kreuzberg,11002002,Kreuzberg,202,...,13.405233,POINT (13.4052329 52.4956842),<NA>,"Mo-Sa 10:00-12:00, Mo 17:00-19:00, Tu,We,Fr 16...",Mon–Sat,<NA>,<NA>,<NA>,<NA>,no
1,299795048,Dr. med. vet. Elke Hartwig,Straße 48,67,13125.0,Berlin,Pankow,11003003,Karow,305,...,13.479555,POINT (13.4795548 52.60628629999999),<NA>,"Mo,Tu,Th,Fr 10:00-12:00, Mo-Fr 15:00-18:00",Mon–Fri,+49 30 9437820,<NA>,http://www.tierarztpraxis-hartwig.de/,phone: +49 30 9437820; web: http://www.tierarz...,limited
2,347294456,Tierarztpraxis Dr. Bernhard Sörensen,Königsberger Straße,36,12207.0,Berlin,Steglitz-Zehlendorf,11006006,Lichterfelde,602,...,13.320133,POINT (13.3201326 52.4297216),<NA>,"Mo-Fr 09:00-20:00; Sa, Su 10:00-18:00",Mon–Sun,+49 30 7738321,<NA>,https://www.tierarztpraxis-soerensen.de/,phone: +49 30 7738321; web: https://www.tierar...,yes
3,394867279,Tierarztpraxis Jeanette Koepsel,<NA>,<NA>,<NA>,<NA>,Spandau,11005005,Siemensstadt,503,...,13.270573,POINT (13.2705734 52.5351995),<NA>,<NA>,None,<NA>,<NA>,<NA>,<NA>,<NA>
4,411550894,Kleintierarztpraxis Berlin Kaulsdorf,Planitzstraße,19,12621.0,Berlin,Marzahn-Hellersdorf,11010010,Kaulsdorf,1003,...,13.589635,POINT (13.5896353 52.50951139999999),<NA>,"Mo-Fr 09:00-19:00 open ""tel. Terminvereinbarun...",Mon–Sat,+49 30 53018585,info@tierarzt-kaulsdorf.de,https://www.tierarzt-kaulsdorf.de/,phone: +49 30 53018585; email: info@tierarzt-k...,<NA>


## 3. Optional: Reverse geocoding with Nominatim for missing addresses

For clinics where the address is missing but coordinates are available, we can
use Nominatim to backfill `addr_street`, `addr_housenumber`, `addr_postcode`,
and `addr_city`.

To respect Nominatim's rate limits, we:

- use a custom `user_agent`,
- add a 1-second delay between calls,
- by default, limit the number of records to geocode.

In [4]:
# 3. Optional: Reverse geocoding with Nominatim for missing addresses

# We try to import geopy. If it's not available, we skip this step gracefully.
try:
    from geopy.geocoders import Nominatim
    from time import sleep

    geolocator = Nominatim(user_agent="berlin_vet_clinics_reverse_geocoder")
    NOMINATIM_AVAILABLE = True
except ImportError:
    geolocator = None
    NOMINATIM_AVAILABLE = False
    print("geopy is not installed – skipping reverse geocoding step.")


def reverse_geocode_address(lat, lon):
    """
    Reverse geocode to retrieve address fields from coordinates.
    Returns (street, housenumber, postcode, city) or (None, ...).
    """
    if not NOMINATIM_AVAILABLE:
        return None, None, None, None

    try:
        location = geolocator.reverse(
            (lat, lon),
            exactly_one=True,
            language="de"
        )
        # Be gentle with the public API
        sleep(1)
        if location is None:
            return None, None, None, None

        addr = location.raw.get("address", {})
        street = (
            addr.get("road")
            or addr.get("pedestrian")
            or addr.get("footway")
            or addr.get("residential")
        )
        housenumber = addr.get("house_number")
        postcode = addr.get("postcode")
        city = (
            addr.get("city")
            or addr.get("town")
            or addr.get("village")
            or addr.get("suburb")
        )
        return street, housenumber, postcode, city
    except Exception:
        return None, None, None, None


if NOMINATIM_AVAILABLE:
    # Select candidates with missing address but valid coordinates
    mask_missing_addr = (
        df_clean["addr_street"].isna()
        & df_clean["addr_postcode"].isna()
        & df_clean["addr_city"].isna()
        & df_clean["latitude"].notna()
        & df_clean["longitude"].notna()
    )

    candidates = df_clean[mask_missing_addr].copy()
    print(f"Rows with missing address and valid coords: {len(candidates)}")

    # Limit the number of calls to avoid hitting rate limits
    MAX_REVERSE_GEOCODES = 50

    for idx, row in candidates.head(MAX_REVERSE_GEOCODES).iterrows():
        street, hn, pc, city = reverse_geocode_address(row["latitude"], row["longitude"])

        if street and pd.isna(df_clean.at[idx, "addr_street"]):
            df_clean.at[idx, "addr_street"] = street
        if hn and pd.isna(df_clean.at[idx, "addr_housenumber"]):
            df_clean.at[idx, "addr_housenumber"] = hn
        if pc and pd.isna(df_clean.at[idx, "addr_postcode"]):
            df_clean.at[idx, "addr_postcode"] = pc
        if city and pd.isna(df_clean.at[idx, "addr_city"]):
            df_clean.at[idx, "addr_city"] = city

    print("Reverse geocoding step finished (limited to", MAX_REVERSE_GEOCODES, "rows).")
else:
    print("Reverse geocoding section skipped because geopy is missing.")

Rows with missing address and valid coords: 48
Reverse geocoding step finished (limited to 50 rows).


In [5]:
mask_missing_addr = (
    df_clean["addr_street"].isna()
    & df_clean["addr_postcode"].isna()
    & df_clean["addr_city"].isna()
    & df_clean["latitude"].notna()
    & df_clean["longitude"].notna()
)

df_clean.loc[mask_missing_addr, [
    "id", "clinic_name",
    "addr_street", "addr_housenumber", "addr_postcode", "addr_city",
    "latitude", "longitude"
]]

,id,clinic_name,addr_street,addr_housenumber,addr_postcode,addr_city,latitude,longitude


## 4. Full address, data source and quality flag

In [6]:
# 4.1 Full address (now including any Nominatim-enriched components)
df_addr_tmp = df_clean[
    ["addr_street", "addr_housenumber", "addr_postcode", "addr_city"]
]
df_clean["full_address"] = df_addr_tmp.apply(build_full_address, axis=1).astype("string")

# 4.2 Data source description for provenance
df_clean["data_source"] = (
    "OSM amenity=veterinary, Berlin, fetched via OSMNX latest snapshot"
)

# 4.3 Keep OSM source identifier
df_clean["source_osm_id"] = df["source_osm_id"].astype("string")

# 4.4 Quality flag: mark clinics with at least a minimum amount of information
df_clean["has_minimum_info"] = (
    df_clean["clinic_name"].notna()
    | df_clean["full_address"].notna()
    | df_clean["phone_main"].notna()
    | df_clean["website_main"].notna()
)

df_clean[["id", "clinic_name", "full_address", "has_minimum_info"]].head()

,id,clinic_name,full_address,has_minimum_info
0,268917040,Tierarztpraxis am Urban,"Baerwaldstraße 69, 10961.0 Berlin",True
1,299795048,Dr. med. vet. Elke Hartwig,"Straße 48 67, 13125.0 Berlin",True
2,347294456,Tierarztpraxis Dr. Bernhard Sörensen,"Königsberger Straße 36, 12207.0 Berlin",True
3,394867279,Tierarztpraxis Jeanette Koepsel,"Wernerwerkdamm 27, 13629 Berlin",True
4,411550894,Kleintierarztpraxis Berlin Kaulsdorf,"Planitzstraße 19, 12621.0 Berlin",True


## 5. Normalise NULLs vs empty strings

We ensure key fields use NULL (`NaN` / `pd.NA`) for missing values instead of
empty strings, to keep SQL filters simple and consistent.

In [7]:
# 5. Normalise NULLs vs empty strings

cols_to_null_empty = [
    "clinic_name",
    "addr_street",
    "addr_housenumber",
    "addr_postcode",
    "addr_city",
    "full_address",
    "district",
    "district_id",
    "neighborhood",
    "neighborhood_id",
    "lor_id",
    "services_offered",
    "operating_hours",
    "operating_days",
    "phone_main",
    "email_main",
    "website_main",
    "contact_info",
    "accessibility_features",
    "data_source",
]

for col in cols_to_null_empty:
    if col in df_clean.columns:
        df_clean[col] = (
            df_clean[col]
            .astype("string")
            .replace(r"^\s*$", pd.NA, regex=True)
        )

## 6. Quick data quality checks

In [8]:
print("Null counts per column:")
print(df_clean.isna().sum())

print("\nEmpty or NULL clinic_name rows:", df_clean["clinic_name"].isna().sum())
print("NULL district rows:", df_clean["district"].isna().sum())
print("NULL neighborhood rows:", df_clean["neighborhood"].isna().sum())
print("NULL latitude rows:", df_clean["latitude"].isna().sum())
print("NULL longitude rows:", df_clean["longitude"].isna().sum())

print("\nRecord quality (has_minimum_info value counts):")
print(df_clean["has_minimum_info"].value_counts(dropna=False))

Null counts per column:
id                          0
clinic_name                 4
addr_street                 0
addr_housenumber           37
addr_postcode              12
addr_city                  14
district                    0
district_id                 0
neighborhood                0
neighborhood_id             0
lor_id                      0
latitude                    0
longitude                   0
geometry                    0
services_offered          173
operating_hours            50
operating_days             56
phone_main                 78
email_main                145
website_main               66
contact_info               53
accessibility_features     93
full_address                0
data_source                 0
source_osm_id               0
has_minimum_info            0
dtype: int64

Empty or NULL clinic_name rows: 4
NULL district rows: 0
NULL neighborhood rows: 0
NULL latitude rows: 0
NULL longitude rows: 0

Record quality (has_minimum_info value counts):
has_mi

## 7. Export v1 cleaned table

In [10]:
from pathlib import Path

output_v1_csv = Path("cache/vet_clinics_berlin_clean_latest_v1.csv")
output_v1_csv.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(output_v1_csv, index=False)

print("Exported cleaned v1 file to:")
print(" -", output_v1_csv)
print("Shape:", df_clean.shape)

Exported cleaned v1 file to:
 - cache/vet_clinics_berlin_clean_latest_v1.csv
Shape: (175, 26)
